In [ ]:
# Imports
import glob as glob
import pandas as pd
import numpy as np

In [ ]:
# We will import data; and we aggregate data to be our desired features.
# Yes current feature extraction is manual, but we can automate it in the future. We will use the data to train our model.

# Clean and pivot all *_merged CSV files under bookDepth_data/{pair} then combine into one cleaned file
import glob
from pathlib import Path
import pandas as pd

merged_files = sorted(glob.glob("bookDepth_data/*/*_merged*.csv"))
print(f"Found {len(merged_files)} merged files in bookDepth_data")

all_cleaned = []
for merged_file in merged_files:
    pair = Path(merged_file).parts[-2]
    print(f"Processing {merged_file} (pair={pair})")

    df = pd.read_csv(merged_file, parse_dates=["timestamp"])
    if df.empty:
        print("  empty, skipping")
        continue

    pivoted = df.pivot_table(
        index="timestamp",
        columns="percentage",
        values=["depth", "notional"],
        aggfunc="first"
    )
    pivoted.columns = [f"{pair}_{v}_{k}" for v, k in pivoted.columns]
    pivoted = pivoted.reset_index()

    all_cleaned.append(pivoted)
    print(f"  pivoted rows={len(pivoted)}, cols={len(pivoted.columns)}")

if all_cleaned:
    combined = pd.concat(all_cleaned, ignore_index=True, sort=False)
    combined = combined.sort_values(["timestamp"]).reset_index(drop=True)
    out_file = "bookDepth_data/all_pairs_cleaned.csv"
    Path("bookDepth_data").mkdir(parents=True, exist_ok=True)
    combined.to_csv(out_file, index=False)
    print(f"Combined cleaned file saved: {out_file} rows={len(combined)}, cols={len(combined.columns)}")
else:
    print("No cleaned rows to combine.")

Found 1 merged files in bookDepth_data
Processing bookDepth_data\ETHUSDT\ETHUSDT_merged.csv (pair=ETHUSDT)
  pivoted rows=18432, cols=25
Combined cleaned file saved: bookDepth_data/all_pairs_cleaned.csv rows=18432, cols=25


In [ ]:
# We will do some feature engineering to create features that can be used for later inference.
try:
    df = pd.read_csv("bookDepth_data/all_pairs_cleaned.csv", parse_dates=["timestamp"])
except FileNotFoundError:
    print("File not found. Please check the file path.")
    raise
except pd.errors.ParserError as e:
    print("CSV parse error:", e)
    raise
print(f"Loaded cleaned data: rows={len(df)}, cols={len(df.columns)}")
BASE_SIZE = 24  # Excluding timestamp, we have 24 columns per pair (12 percentage levels * 2 features each)
# Build pairs from each chunk of BASE_SIZE columns after timestamp.
pair_names = []
num_data_cols = len(df.columns) - 1  # exclude timestamp
if num_data_cols % BASE_SIZE != 0:
    raise ValueError(f"Expected data columns to be a multiple of BASE_SIZE ({BASE_SIZE}), got {num_data_cols}")
for chunk_idx in range(num_data_cols // BASE_SIZE):
    col_idx = 1 + chunk_idx * BASE_SIZE
    first_col = df.columns[col_idx]
    pair_name = first_col.split("_")[0]
    pair_names.append(pair_name)

Trading_Pairs = pair_names
print(f"Trading pairs: {Trading_Pairs}")

# 1) Compute per-pair depth/notional totals and imbalance ratios (base features)
for pair in Trading_Pairs:
    depth_neg_cols = [c for c in df.columns if c.startswith(pair + "_depth_-")]
    depth_pos_cols = [c for c in df.columns if c.startswith(pair + "_depth_") and not c.startswith(pair + "_depth_-")]
    df[f"{pair}_total_depth_neg"] = df[depth_neg_cols].sum(axis=1)
    df[f"{pair}_total_depth_pos"] = df[depth_pos_cols].sum(axis=1)
    df[f"{pair}_total_depth"] = df[f"{pair}_total_depth_neg"] + df[f"{pair}_total_depth_pos"]
    df[f"{pair}_depth_imbalance"] = df[f"{pair}_total_depth_pos"] - df[f"{pair}_total_depth_neg"]
    df[f"{pair}_depth_imbalance_ratio"] = df[f"{pair}_depth_imbalance"] / df[f"{pair}_total_depth"].replace(0, np.nan)

    notional_neg_cols = [c for c in df.columns if c.startswith(pair + "_notional_-")]
    notional_pos_cols = [c for c in df.columns if c.startswith(pair + "_notional_") and not c.startswith(pair + "_notional_-")]
    df[f"{pair}_total_notional_neg"] = df[notional_neg_cols].sum(axis=1)
    df[f"{pair}_total_notional_pos"] = df[notional_pos_cols].sum(axis=1)
    df[f"{pair}_total_notional"] = df[f"{pair}_total_notional_neg"] + df[f"{pair}_total_notional_pos"]
    df[f"{pair}_notional_imbalance"] = df[f"{pair}_total_notional_pos"] - df[f"{pair}_total_notional_neg"]
    df[f"{pair}_notional_imbalance_ratio"] = df[f"{pair}_notional_imbalance"] / df[f"{pair}_total_notional"].replace(0, np.nan)

# 2) Add T vs T-1 immediate velocity-like change for imbalance ratio and notional totals.
for pair in Trading_Pairs:
    df[f"{pair}_depth_imbalance_ratio_delta"] = df[f"{pair}_depth_imbalance_ratio"].diff()
    df[f"{pair}_notional_imbalance_ratio_delta"] = df[f"{pair}_notional_imbalance_ratio"].diff()
    df[f"{pair}_total_notional_delta"] = df[f"{pair}_total_notional"].diff()

# 3) Rolling regime normalization (larger window: 60 or 240 minutes)
roll_window = 60
for pair in Trading_Pairs:
    df[f"{pair}_log_notional"] = np.log1p(df[f"{pair}_total_notional"])
    roll_mean = df[f"{pair}_log_notional"].rolling(roll_window, min_periods=1).mean()
    roll_std = df[f"{pair}_log_notional"].rolling(roll_window, min_periods=1).std().replace(0, np.nan)
    df[f"{pair}_notional_z"] = (df[f"{pair}_log_notional"] - roll_mean) / roll_std

    regime_window = 240
    avg = df[f"{pair}_log_notional"].rolling(regime_window, min_periods=1).mean()
    std = df[f"{pair}_log_notional"].rolling(regime_window, min_periods=1).std().replace(0, np.nan)
    df[f"{pair}_notional_regime_z"] = (df[f"{pair}_log_notional"] - avg) / std

# 4) Drop raw columns and keep ratios + standardized features.
cols_to_drop = []
for pair in Trading_Pairs:
    cols_to_drop.extend([
        f"{pair}_total_depth", f"{pair}_total_depth_pos", f"{pair}_total_depth_neg",
        f"{pair}_total_notional", f"{pair}_log_notional"
    ])

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f"Feature engineering completed. Total columns now: {len(df.columns)}")
    

Loaded cleaned data: rows=18432, cols=25
Columns: ['timestamp', 'ETHUSDT_depth_-5.0', 'ETHUSDT_depth_-4.0', 'ETHUSDT_depth_-3.0', 'ETHUSDT_depth_-2.0', 'ETHUSDT_depth_-1.0', 'ETHUSDT_depth_-0.2', 'ETHUSDT_depth_0.2', 'ETHUSDT_depth_1.0', 'ETHUSDT_depth_2.0', 'ETHUSDT_depth_3.0', 'ETHUSDT_depth_4.0', 'ETHUSDT_depth_5.0', 'ETHUSDT_notional_-5.0', 'ETHUSDT_notional_-4.0', 'ETHUSDT_notional_-3.0', 'ETHUSDT_notional_-2.0', 'ETHUSDT_notional_-1.0', 'ETHUSDT_notional_-0.2', 'ETHUSDT_notional_0.2', 'ETHUSDT_notional_1.0', 'ETHUSDT_notional_2.0', 'ETHUSDT_notional_3.0', 'ETHUSDT_notional_4.0', 'ETHUSDT_notional_5.0']
Trading pairs: ['ETHUSDT']
